In [1]:
#| default_exp dyalog

# Dyalog sessions
> Run Dyalog APL through RIDE for reference checks.

Import `Apl` and `AplError` from `aplnb.dyalog`. Sessions return Dyalog output or JSON-converted Python values, not bAsedPL arrays. Importing this module does not start Dyalog or register notebook magics.

Install Dyalog separately. The examples use a local interpreter and run with `nbdev-test --flags dyalog`. The default test run skips them.

In [2]:
from fastcore.test import *

In [3]:
#| export
import atexit, json, socket, subprocess
from shutil import which
from fastcore.utils import *
from aplnb.core import AplOut

## Connecting

In [4]:
#| export
def find_dyalog():
    "Locate the Dyalog interpreter binary"
    if p:=which('mapl') or which('dyalog'): return p
    apps = sorted(Path('/Applications').glob('Dyalog-*.app'))
    if apps: return str(apps[-1]/'Contents/Resources/Dyalog/mapl')
    vers = sorted(Path('/opt/mdyalog').glob('*/*/*/mapl'))
    if vers: return str(vers[-1])
    raise FileNotFoundError('Dyalog APL not found: install it from dyalog.com')

def start_dyalog(
    dyalog=None, # Path to the interpreter binary; `find_dyalog()` result if None
    timeout=10, # Socket timeout during the startup handshake, in seconds
):
    "Spawn a Dyalog interpreter that connects back to us over RIDE; return `(socket,Popen)`"
    if not dyalog: dyalog = find_dyalog()
    lsn = socket.create_server(('127.0.0.1', 0))
    port = lsn.getsockname()[1]
    env = os.environ | dict(RIDE_INIT=f'CONNECT:127.0.0.1:{port}', RIDE_SPAWNED='1', MAXAPLCORES=os.environ.get('MAXAPLCORES','0'),
        DYALOGQUIETUCMDBUILD='1', DYALOG_LINEEDITOR_MODE='1', ENABLE_CEF='0', LOG_FILE_INUSE='0')
    dn = subprocess.DEVNULL
    proc = subprocess.Popen([dyalog], env=env, stdin=dn, stdout=dn, stderr=dn)
    lsn.settimeout(10)
    sock,_ = lsn.accept()
    lsn.close()
    sock.settimeout(timeout)
    return sock,proc

`find_dyalog` checks the command path and standard macOS/Linux installation locations. Pass `dyalog=` to `Apl` to choose another executable. `timeout` bounds the startup handshake, not evaluation.

In [5]:
#| export
def ride_send(sock, msg):
    "Send one RIDE message: a handshake `str`, or a `[cmd,args]` list sent as JSON"
    if not isinstance(msg,str): msg = json.dumps(msg, separators=(',',':'))
    b = ('RIDE'+msg).encode()
    sock.sendall((len(b)+4).to_bytes(4,'big')+b)

def _recvall(sock, n):
    parts = []
    while n:
        b = sock.recv(n)
        if not b: raise ConnectionError('Dyalog closed the connection')
        parts.append(b)
        n -= len(b)
    return b''.join(parts)

def ride_recv(sock):
    "Receive one RIDE message, JSON-decoded unless it's a handshake string"
    hdr = _recvall(sock, 8)
    assert hdr[4:8]==b'RIDE', f"Bad RIDE header: {hdr}"
    msg = _recvall(sock, int.from_bytes(hdr[:4],'big')-8).decode()
    msg = re.sub(r'[\x00-\x1f]', lambda m: f'\\u{ord(m[0]):04x}', msg)
    return json.loads(msg) if msg[0]=='[' else msg

RIDE uses length-prefixed messages over a local socket. `ride_run` collects session output until Dyalog is ready for another expression. Interactive and incomplete-input prompts are handled by `Apl.run`.

In [6]:
#| export
class AplError(Exception):
    "A Dyalog diagnostic; `reset` means the interpreter was replaced and workspace state lost"
    def __init__(self, msg, reset=False):
        super().__init__(msg)
        self.reset = reset

class AplPrompt(Exception):
    "A non-ready prompt: 2=⎕ input, 3=incomplete input, 4=⍞ input"
    def __init__(self, ptype):
        super().__init__(f'prompt type {ptype}')
        self.ptype = ptype

def ride_run(sock, code):
    "Run APL lines; return `(output,errno)` once the session is ready again"
    lines = [l for l in code.splitlines() if l.strip()]
    ride_send(sock, ['Execute',{'text':'\n'.join(lines)+'\n','trace':0}])
    out,err,pending = '',0,len(lines)
    while True:
        m,a = ride_recv(sock)
        if m=='AppendSessionOutput':
            if a['type']==14: pending -= 1
            elif a['type']!=1: out += a['result']
        elif m=='HadError': err = a['error']
        elif m=='SetPromptType' and a['type']!=0 and not pending:
            if a['type']!=1: raise AplPrompt(a['type'])
            return out,err

In [7]:
#| export
class Apl:
    "A Dyalog APL session over the RIDE protocol"
    def __init__(self, dyalog=None, timeout=10):
        store_attr()
        self._connect()

    def _connect(self):
        self.sock,self.proc = start_dyalog(self.dyalog, self.timeout)
        assert ride_recv(self.sock)=='SupportedProtocols=2'
        ride_send(self.sock, 'SupportedProtocols=2')
        ride_send(self.sock, 'UsingProtocol=2')
        assert ride_recv(self.sock)=='UsingProtocol=2'
        ride_send(self.sock, ['Identify',{'apiVersion':1,'identity':1}])
        self.info = ride_recv(self.sock)[1]
        ride_send(self.sock, ['SetPW',{'pw':32767}])
        atexit.register(self.close)
        while True:
            m,a = ride_recv(self.sock)
            if m=='SetPromptType' and a['type']==1: break
        self.sock.settimeout(None)

@patch
def close(self:Apl):
    "Shut down the interpreter and close the connection"
    atexit.unregister(self.close)
    try: ride_send(self.sock, ['Exit',{'code':0}])
    except OSError: pass
    self.sock.close()
    try: self.proc.wait(3)
    except subprocess.TimeoutExpired:
        self.proc.kill()
        self.proc.wait()

@patch
def __enter__(self:Apl): return self

@patch
def __exit__(self:Apl, *args): self.close()

## Evaluating APL

In [8]:
#| export
@patch
def run(self:Apl, code):
    "Run `code`, returning session output; raises `AplError` on APL errors"
    try: out,err = ride_run(self.sock, code)
    except AplPrompt as e:
        if e.ptype in (2,4):
            ride_run(self.sock, '→' if e.ptype==2 else '')
            raise AplError('Input via ⎕ or ⍞ is not supported in aplnb') from None
        self.close()
        self._connect()
        raise AplError('Incomplete input wedged the Dyalog interpreter; started a fresh session (workspace lost). See Dyalog/ride#1401',
            reset=True) from None
    if err: raise AplError(out)
    return out

`run` returns session output as text. Assignments persist across calls:

In [ ]:
#| dyalog
dyalog = Apl()
out = dyalog.run('x←1 2 3\n+/x')
test_eq(out.strip(), '6')
out

'6\n'

In [9]:
#| export
@patch
def __call__(self:Apl, code):
    "Run `code`, returning displayable session output, or None if there is none"
    return AplOut(self.run(code)) or None

Calling a session returns `AplOut`, the same text with notebook display support. A silent assignment returns `None`:

In [ ]:
#| dyalog
test_is(dyalog('quiet←7'), None)
dyalog('2×x')

2 4 6

## Python values

In [10]:
#| export
@patch
def pyval(self:Apl, expr):
    "Evaluate `expr` and return its JSON-converted Python value"
    return json.loads(self.run(f"1(⎕JSON⍠'HighRank' 'Split')({expr})"))

`pyval` uses Dyalog's JSON conversion. Scalars become Python values and numeric arrays become lists. This is the interface used by the reference checker:

In [ ]:
#| dyalog
test_eq(dyalog.pyval('+/x'), 6)
matrix = dyalog.pyval('2 3⍴⍳6')
test_eq(matrix, [[1,2,3], [4,5,6]])
matrix

[[1, 2, 3], [4, 5, 6]]

In [11]:
#| export
def _apljson(v):
    "An APL expression that evaluates to the Python value `v`"
    return "⎕JSON'" + json.dumps(v).replace("'", "''") + "'"

@patch
def __getitem__(self:Apl, expr): return self.pyval(expr)

@patch
def __setitem__(self:Apl, nm, v): self.run(f'{nm}←{_apljson(v)}')

Square brackets read an expression or assign a JSON-compatible Python value. Quotes in strings are escaped for APL:

In [ ]:
#| dyalog
dyalog['values'] = [3,1,4]
dyalog['message'] = "can't"
test_eq(dyalog['message'], "can't")
test_eq(dyalog['values'], [3,1,4])
dyalog['values']

[3, 1, 4]

In [12]:
#| export
@patch
def fn(self:Apl, code):
    "A Python callable applying APL function `code` monadically or dyadically"
    def f(*args):
        if len(args)==1: return self.pyval(f'({code}){_apljson(args[0])}')
        a,w = args
        return self.pyval(f'({_apljson(a)})({code}){_apljson(w)}')
    return f

`fn` converts Python arguments through JSON and evaluates the function expression on each call. One argument supplies `⍵`; two supply `⍺` and `⍵`:

In [ ]:
#| dyalog
mean = dyalog.fn('{(+/⍵)÷≢⍵}')
test_eq(mean([1,2,3]), 2)
add = dyalog.fn('+')
test_eq(add([1,2,3], 10), [11,12,13])
add([1,2,3], 10)

[11, 12, 13]

## Errors and lifetime

Dyalog errors raise this module's `AplError`, which is separate from bAsedPL's exception. Completed assignments remain in the session:

In [ ]:
#| dyalog
with expect_fail(AplError, contains='DOMAIN ERROR'): dyalog.run('saved←42 ⋄ 1÷0')
test_eq(dyalog['saved'], 42)
dyalog['saved']

42

Interactive `⎕` and `⍞` input is rejected. An incomplete-input prompt restarts the interpreter and raises `AplError` with `reset=True`; that restart loses the workspace.

Call `close()` when finished, or use a context manager for a bounded reference check:

In [ ]:
#| dyalog
dyalog.close()
with Apl() as reference: total = reference.pyval('+/⍳10')
test_eq(total, 55)
total

55